# Firm-Wide Economic Capital Demo

**Economic Capital Simulator** – Full Firm-Wide Aggregation  
Ajayvir Khara | Passed FRM Part I and Part II | January 2026

This notebook demonstrates the **complete firm-wide Economic Capital simulation** across:

- **Market Risk** – Multi-asset VaR/ES with Student-t shocks
- **Credit Risk** – Counterparty exposure + WWR-aware EL/UL
- **Operational Risk** – LDA with hybrid severity + expert judgment overlay

Key features:
- Full 750,000-path Monte Carlo with **t-copula** (df=3) for realistic tail dependence
- Diversification benefit calculation
- Euler (marginal) allocation of total EC to each risk type
- Automated generation of **regulatory-grade Excel report**

**Expected runtime**: ~3–6 minutes on a standard laptop

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import os
import warnings
import glob
import time

warnings.filterwarnings("ignore", category=UserWarning)

# Styling
plt.style.use("seaborn-v0_8-pastel")
sns.set_palette("husl")
plt.rcParams["figure.figsize"] = (14, 7)
%matplotlib inline

# Project root detection
notebook_dir = Path.cwd().resolve()
if notebook_dir.name == "notebooks":
    root = notebook_dir.parent
else:
    root = notebook_dir
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # Force this into the front of the path
os.chdir(root)  # Change the working directory to the root

print(f"Current Working Directory fixed to: {os.getcwd()}")

In [ ]:
from econ_capital import run_full_simulation

## Run Full Firm-Wide Simulation

In [ ]:
%%time

from datetime import datetime

print(f"Starting firm-wide simulation at {datetime.now():%Y-%m-%d %H:%M:%S}\n")

results = run_full_simulation()

EL_total = results["EL_total"]
UL_portfolio = results["UL_portfolio"]
EC_total = results["EC_total"]
div_benefit = results["diversification_benefit"]
marginal = results["marginal_contributions"]
individual = results["individual_risks"]

## Summary Results

In [ ]:
from econ_capital.market_risk.data_loaders import load_real_risk_factors

# Manually fetch the data to inspect it
df_debug = load_real_risk_factors(start="2021-01-01", end="2026-01-01")

print("Checking for missing values:")
print(df_debug.isnull().sum())

print("\nChecking variance (should not be 0):")
print(df_debug.var())

In [ ]:
print("=" * 70)
print("FIRM-WIDE ECONOMIC CAPITAL SUMMARY")
print("=" * 70)
print(f"{'Expected Loss (EL)':<35} : £{EL_total:>18,.0f}")
print(f"{'Portfolio Unexpected Loss (UL)':<35} : £{UL_portfolio:>18,.0f}")
print(f"{'Total Economic Capital (99.9%)':<35} : £{EC_total:>18,.0f}")
print(
    f"{'Diversification Benefit':<35} : £{div_benefit:>18,.0f} ({div_benefit / EC_total * 100:.1f}%)"
)
print("\nMarginal Contributions to Total EC:")
for risk, contrib in marginal.items():
    pct = contrib / EC_total * 100 if EC_total > 0 else 0
    print(f"   • {risk:<12}: £{contrib:>15,.0f} ({pct:>6.1f}%)")
print("=" * 70)

## Visualisation: Marginal Contribution Breakdown

In [ ]:
contrib_df = pd.DataFrame(
    {
        "Risk Type": list(marginal.keys()),
        "Marginal EC (£m)": [v / 1e6 for v in marginal.values()],
        "% of Total": [
            (v / EC_total * 100) if EC_total > 0 else 0 for v in marginal.values()
        ],
    }
).sort_values("Marginal EC (£m)", ascending=False)

plt.figure(figsize=(12, 6))
bars = sns.barplot(data=contrib_df, x="Risk Type", y="Marginal EC (£m)")
plt.title("Marginal Contribution to Firm-Wide Economic Capital")
plt.ylabel("Marginal EC (£ million)")
plt.xlabel("Risk Type")
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", alpha=0.3)

# Add percentage labels
for bar, pct in zip(bars.patches, contrib_df["% of Total"]):
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height + height * 0.02,
        f"{pct:.1f}%",
        ha="center",
        va="bottom",
    )

plt.tight_layout()
plt.show()

## Dashboard: Firm-Wide Economic Capital at a Glance

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import numpy as np
from scipy.stats import t as tdist

# =============================================================================
# FIRM-WIDE ECONOMIC CAPITAL DASHBOARD
# Visualises aggregated EC output across Market, Credit, and Operational Risk.
# Requires: EC_total, EL_total, UL_portfolio, div_benefit, marginal, individual
# =============================================================================

# --- Colour palette: dark terminal aesthetic consistent with risk system UX ---
BG, PANEL, BORDER = "#0d0f14", "#13161e", "#1e2330"
GOLD, GREEN, RED, BLUE, MUTED, WHITE = (
    "#c9a84c",
    "#4ecf8a",
    "#e05c5c",
    "#4a9eda",
    "#6b7a99",
    "#f0f2f8",
)

# Risk-type colour mapping - consistent across all panels for visual coherence
RISK_CLR = {"Market": BLUE, "Credit": RED, "OpRisk": GOLD}

# Apply global rcParams - overrides seaborn defaults set earlier in the notebook
plt.rcParams.update(
    {
        "font.family": "monospace",  # Terminal-style font; reinforces quantitative aesthetic
        "text.color": WHITE,
        "axes.facecolor": PANEL,  # Dark panel background for all axes
        "figure.facecolor": BG,  # Outer figure background
        "axes.edgecolor": BORDER,
        "axes.labelcolor": MUTED,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "grid.color": BORDER,
        "grid.linewidth": 0.6,
    }
)

# =============================================================================
# DATA PREPARATION
# All monetary figures converted to £ millions for readability across panels.
# =============================================================================

risk_labels = list(marginal.keys())  # Ordered list of risk types from simulation output
marginal_m = [
    marginal[k] / 1e6 for k in risk_labels
]  # Euler-allocated marginal EC per risk type (£m)
total_m = [
    individual[k]["Total_Standalone"] / 1e6 for k in risk_labels
]  # Standalone (undiversified) EC per risk type (£m)
el_by_risk = [
    individual[k]["EL"] / 1e6 for k in risk_labels
]  # Expected Loss per risk type - used in capital intensity ratio
colors = [RISK_CLR[r] for r in risk_labels]  # Colour array aligned to risk_labels order

ec_m = EC_total / 1e6  # Total diversified firm-wide EC (£m)
el_m = EL_total / 1e6  # Total firm-wide Expected Loss (£m)
ul_m = UL_portfolio / 1e6  # Portfolio Unexpected Loss - proxy for volatility (£m)
div_m = div_benefit / 1e6  # Absolute diversification benefit (£m) = naive sum − EC
naive_m = sum(total_m)  # Undiversified (sum-of-parts) EC - benchmark for div benefit
div_pct = (
    div_benefit / EC_total * 100
)  # Diversification benefit as % of undiversified total

# =============================================================================
# FIGURE INITIALISATION
# 3-row × 4-column GridSpec layout:
#   Row 0: KPI summary tiles
#   Row 1: Waterfall | Donut | Standalone vs Marginal
#   Row 2: P&L distribution | Capital intensity | Confidence sensitivity
# =============================================================================

fig = plt.figure(figsize=(22, 13), facecolor=BG)

# Dashboard header - firm-wide scope and simulation parameters
fig.text(
    0.015,
    0.975,
    "ECONOMIC CAPITAL DASHBOARD",
    color=GOLD,
    fontsize=20,
    fontweight="bold",
    va="top",
    fontfamily="monospace",
)
fig.text(
    0.015,
    0.945,
    "Firm-Wide  |  99.9% Confidence  |  t-Copula (df=3)  |  750,000 Paths",
    color=MUTED,
    fontsize=9,
    va="top",
    fontfamily="monospace",
)

# GridSpec with explicit margins - prevents label clipping at figure edges
gs = gridspec.GridSpec(
    3,
    4,
    figure=fig,
    left=0.05,
    right=0.97,
    top=0.90,
    bottom=0.06,
    hspace=0.58,
    wspace=0.38,
)

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================


def styled_ax(ax, title=""):
    """Apply consistent dark-theme styling to any axis."""
    ax.set_facecolor(PANEL)
    for sp in ax.spines.values():
        sp.set_edgecolor(BORDER)
        sp.set_linewidth(0.8)
    if title:
        ax.set_title(
            title,
            color=GOLD,
            fontsize=9,
            fontweight="bold",
            loc="left",
            pad=8,
            fontfamily="monospace",
        )
    ax.tick_params(labelsize=8)
    return ax


def kpi_box(ax, label, value, sub="", color=GOLD):
    """Render a borderless KPI tile with large metric value, label, and subtitle."""
    ax.set_facecolor(PANEL)
    for sp in ax.spines.values():
        sp.set_edgecolor(color)
        sp.set_linewidth(1.8)  # Coloured border signals metric category at a glance
    ax.set_xticks([])
    ax.set_yticks([])
    ax.text(
        0.5,
        0.65,
        value,
        ha="center",
        va="center",
        transform=ax.transAxes,
        color=color,
        fontsize=22,
        fontweight="bold",
        fontfamily="monospace",
    )
    ax.text(
        0.5,
        0.35,
        label,
        ha="center",
        va="center",
        transform=ax.transAxes,
        color=WHITE,
        fontsize=11,
        fontfamily="monospace",
    )
    ax.text(
        0.5,
        0.14,
        sub,
        ha="center",
        va="center",
        transform=ax.transAxes,
        color=MUTED,
        fontsize=9,
        fontfamily="monospace",
    )


# =============================================================================
# ROW 0: KPI SUMMARY TILES
# Four headline metrics providing an at-a-glance firm-wide capital summary.
# =============================================================================

kpi_box(
    fig.add_subplot(gs[0, 0]),
    "Total Economic Capital",
    f"£{ec_m:,.1f}m",
    "99.9% confidence",
    GOLD,
)
kpi_box(
    fig.add_subplot(gs[0, 1]),
    "Diversification Benefit",
    f"£{div_m:,.1f}m",
    f"{div_pct:.1f}% of undiversified EC",
    GREEN,
)
kpi_box(
    fig.add_subplot(gs[0, 2]),
    "Expected Loss (EL)",
    f"£{el_m:,.1f}m",
    "mean annual loss",
    BLUE,
)
kpi_box(
    fig.add_subplot(gs[0, 3]),
    "Unexpected Loss (UL)",
    f"£{ul_m:,.1f}m",
    "portfolio volatility",
    RED,
)

# =============================================================================
# ROW 1, COL 0-1: EC WATERFALL CHART
# Builds from zero through standalone risk-type contributions, then subtracts
# the diversification benefit to arrive at the diversified Total EC.
# The downward diversification bar makes the benefit visually unambiguous.
# =============================================================================

ax_wf = styled_ax(
    fig.add_subplot(gs[1, 0:2]),
    "EC Waterfall: Standalone → Diversification Benefit → Total",
)

# Stacked bar bottoms: each risk bar starts where the previous ended
bottoms = [sum(total_m[:i]) for i in range(len(total_m))] + [ec_m, 0]
vals = total_m + [-div_m, ec_m]  # Negative div_m produces the downward step
bcolors = colors + [GREEN, GOLD]
xlbls = risk_labels + ["Diversification", "Total EC"]

x = np.arange(len(xlbls))
ax_wf.bar(
    x,
    vals,
    bottom=bottoms,
    color=bcolors,
    width=0.55,
    zorder=3,
    alpha=0.88,
    edgecolor=BG,
    linewidth=0.5,
)

# Annotate each bar with its absolute £ value
for xi, (b, v, c) in enumerate(zip(bottoms, vals, bcolors)):
    ax_wf.text(
        xi,
        (b + v if v > 0 else b) + naive_m * 0.012,
        f"£{abs(v):.0f}m",
        ha="center",
        va="bottom",
        color=c,
        fontsize=7.5,
        fontweight="bold",
        fontfamily="monospace",
    )

# Dashed reference line marking the undiversified (sum-of-parts) total
ax_wf.axhline(naive_m, color=MUTED, linewidth=0.8, linestyle="--", zorder=2)
ax_wf.text(
    len(x) - 0.45,
    naive_m + naive_m * 0.008,
    f"Undiversified: £{naive_m:.0f}m",
    color=MUTED,
    fontsize=7,
    ha="right",
    fontfamily="monospace",
)

ax_wf.set_xticks(x)
ax_wf.set_xticklabels(xlbls, rotation=20, ha="right", fontsize=8)
ax_wf.set_ylabel("£ million", fontsize=8)
ax_wf.yaxis.grid(True, zorder=0)

# =============================================================================
# ROW 1, COL 2: MARGINAL EC DONUT CHART
# Shows Euler-allocated marginal contributions as a proportion of Total EC.
# Marginal (not standalone) EC is the correct basis for capital attribution
# as it accounts for diversification within the aggregation.
# =============================================================================

ax_d = styled_ax(fig.add_subplot(gs[1, 2]), "Marginal EC Composition")
ax_d.set_aspect("equal")  # Forces circular (not elliptical) pie rendering

_, _, autotexts = ax_d.pie(
    marginal_m,
    colors=colors,
    autopct="%1.1f%%",
    startangle=90,
    wedgeprops={
        "linewidth": 2,
        "edgecolor": BG,
        "width": 0.52,
    },  # Ring width = 0.52 → donut style
    pctdistance=0.75,
)

for at in autotexts:
    at.set_color(BG)
    at.set_fontsize(8.5)
    at.set_fontfamily("monospace")
    at.set_fontweight("bold")

# Central label - total EC anchors the donut without requiring a separate legend value
ax_d.text(
    0,
    0,
    f"£{ec_m:.0f}m\nTotal EC",
    ha="center",
    va="center",
    color=GOLD,
    fontsize=9.5,
    fontweight="bold",
    fontfamily="monospace",
)

ax_d.legend(
    handles=[mpatches.Patch(color=RISK_CLR[r], label=r) for r in risk_labels],
    loc="lower center",
    bbox_to_anchor=(0.5, -0.18),
    ncol=1,
    fontsize=7.5,
    framealpha=0,
    labelcolor=WHITE,
)

# =============================================================================
# ROW 1, COL 3: STANDALONE VS MARGINAL EC - GROUPED BAR CHART
# Illustrates the diversification discount per risk type: the gap between the
# standalone bar (ignoring cross-risk correlation) and the Euler marginal bar
# (post-diversification allocation) represents each type's correlation benefit.
# =============================================================================

ax_cm = styled_ax(fig.add_subplot(gs[1, 3]), "Standalone vs Marginal EC")
x2 = np.arange(len(risk_labels))
w = 0.32

for bars_x, vals_, alpha_, lbl_ in [
    (
        x2 - w / 2,
        total_m,
        0.45,
        "Standalone",
    ),  # Faded = standalone (pre-diversification)
    (
        x2 + w / 2,
        marginal_m,
        0.92,
        "Marginal",
    ),  # Solid  = marginal  (post-diversification)
]:
    bs = ax_cm.bar(
        bars_x,
        vals_,
        width=w,
        color=colors,
        alpha=alpha_,
        label=lbl_,
        edgecolor=BG,
        linewidth=0.5,
    )
    for bar in bs:
        h = bar.get_height()
        ax_cm.text(
            bar.get_x() + bar.get_width() / 2,
            h + 0.5,
            f"{h:.0f}",
            ha="center",
            va="bottom",
            color=WHITE,
            fontsize=6.5,
            fontfamily="monospace",
        )

ax_cm.set_xticks(x2)
ax_cm.set_xticklabels(risk_labels, fontsize=8)
ax_cm.set_ylabel("£ million", fontsize=8)
ax_cm.yaxis.grid(True, zorder=0)
ax_cm.legend(fontsize=7.5, framealpha=0, labelcolor=WHITE, loc="upper right")

# =============================================================================
# ROW 2, COL 0-1: ILLUSTRATIVE PORTFOLIO P&L DISTRIBUTION
# Simulates a parametric P&L distribution using t(df=4) scaled to match the
# model's UL output. Fat tails (df=4) are consistent with the simulation's
# Student-t copula assumption. The 99.9% VaR threshold is overlaid to anchor
# the EC figure visually within the loss distribution.
# NOTE: This is illustrative; the actual MC distribution is not stored in memory.
# =============================================================================

ax_pnl = styled_ax(
    fig.add_subplot(gs[2, 0:2]),
    "Illustrative Portfolio P&L Distribution (parametric, t df=4)",
)

# Parametric P&L: scale t-draws to match UL (σ proxy), shift by EL (mean loss)
pnl = np.random.default_rng(42).standard_t(df=4, size=200_000) * (ul_m / 1.65) - el_m
var_999 = np.percentile(pnl, 0.1)  # 99.9th percentile loss = left 0.1% quantile

# Clip to [0.05%, 99.5%] range - removes extreme outliers that distort the x-axis
pnl_clip = pnl[(pnl >= np.percentile(pnl, 0.05)) & (pnl <= np.percentile(pnl, 99.5))]

ax_pnl.hist(pnl_clip, bins=200, color=BLUE, alpha=0.55, density=True, zorder=2)
ylim = ax_pnl.get_ylim()

# VaR threshold line and tail region shading
ax_pnl.axvline(var_999, color=GOLD, linewidth=1.8, linestyle="--", zorder=4)
ax_pnl.fill_betweenx(ylim, pnl_clip.min(), var_999, color=RED, alpha=0.25, zorder=1)
ax_pnl.text(
    var_999 + abs(var_999) * 0.03,
    ylim[1] * 0.88,
    f"EC 99.9%\n£{ec_m:.0f}m",
    color=GOLD,
    fontsize=8,
    fontfamily="monospace",
    va="top",
)

ax_pnl.axvline(
    0, color=MUTED, linewidth=0.8, linestyle=":", zorder=3
)  # Zero P&L reference
ax_pnl.set_xlabel("P&L (£ million)", fontsize=8)
ax_pnl.set_ylabel("Density", fontsize=8)
ax_pnl.set_ylim(ylim)
ax_pnl.yaxis.grid(True, zorder=0)

# =============================================================================
# ROW 2, COL 2: CAPITAL INTENSITY - EC / EL MULTIPLIER
# Measures how many £ of capital are held per £ of expected loss per risk type.
# Uses actual EL figures from the simulation output (not illustrative splits),
# making this a model-grounded efficiency metric rather than a heuristic.
# Higher multipliers indicate capital-intensive, tail-heavy risk profiles.
# =============================================================================

ax_eff = styled_ax(fig.add_subplot(gs[2, 2]), "Capital Intensity: EC / EL Multiplier")

# Ratio of Euler marginal EC to simulated EL - both from model output
efficiency = [m / e if e > 0 else 0 for m, e in zip(marginal_m, el_by_risk)]

bars_e = ax_eff.barh(
    risk_labels,
    efficiency,
    color=colors,
    alpha=0.88,
    edgecolor=BG,
    linewidth=0.5,
    height=0.42,
)
for bar, val in zip(bars_e, efficiency):
    ax_eff.text(
        val + max(efficiency) * 0.03,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.1f}x",
        va="center",
        color=WHITE,
        fontsize=8.5,
        fontfamily="monospace",
        fontweight="bold",
    )

ax_eff.set_xlabel("Marginal EC / EL", fontsize=7.5)
ax_eff.xaxis.grid(True, zorder=0)
ax_eff.invert_yaxis()  # Top-to-bottom ordering matches risk_labels order

# =============================================================================
# ROW 2, COL 3: EC SENSITIVITY TO CONFIDENCE LEVEL
# Scales the base 99.9% EC to alternative regulatory confidence levels using
# t-distribution quantile ratios (df=4, consistent with simulation assumption).
# Covers the range from Basel II internal models (99%) to FRTB stressed (99.97%).
# Y-axis is floored at 75% of the lowest point to prevent visual exaggeration.
# =============================================================================

ax_s = styled_ax(fig.add_subplot(gs[2, 3]), "EC Sensitivity to Confidence Level")

conf_levels = [
    0.95,
    0.975,
    0.99,
    0.999,
    0.9997,
]  # 95% to 99.97% - covers Basel/FRTB range
q_base = tdist.ppf(0.999, df=4)  # Base quantile at 99.9% for scaling
ec_at = [
    ec_m * tdist.ppf(c, df=4) / q_base for c in conf_levels
]  # Linearly scaled EC estimates
x_lbl = [f"{c * 100:.2f}%" for c in conf_levels]

ax_s.plot(x_lbl, ec_at, color=GOLD, marker="o", linewidth=2, markersize=7, zorder=4)
ax_s.fill_between(
    range(len(conf_levels)), ec_at[0] * 0.8, ec_at, color=GOLD, alpha=0.08, zorder=1
)  # Subtle fill; floor prevents misleading zero-base

# Annotate the base case confidence level used in the simulation
ax_s.annotate(
    f"Base: £{ec_m:.0f}m",
    xy=(3, ec_at[3]),
    xytext=(2.1, ec_at[3] + ec_m * 0.07),
    color=GOLD,
    fontsize=7.5,
    fontfamily="monospace",
    arrowprops=dict(arrowstyle="->", color=GOLD, lw=0.9),
)

ax_s.set_ylim(
    bottom=ec_at[0] * 0.75
)  # Y-floor at 75% of lowest EC - avoids misleading scale compression
ax_s.set_ylabel("£ million", fontsize=8)
ax_s.yaxis.grid(True, zorder=0)
ax_s.tick_params(axis="x", labelsize=7.5)

# =============================================================================
# FOOTER & EXPORT
# =============================================================================

fig.text(
    0.97,
    0.005,
    "Ajayvir Khara  |  Economic Capital Simulator  |  t-Copula df=3  |  99.9% VaR",
    ha="right",
    color=MUTED,
    fontsize=7,
    fontfamily="monospace",
)

plt.savefig("ec_dashboard.png", dpi=160, bbox_inches="tight", facecolor=BG)
plt.show()
print("Dashboard saved to ec_dashboard.png")

## Generate Regulatory-Style Firm-Wide Report

In [ ]:
%%time

from econ_capital.firmwide_reporting import generate_firmwide_ec_report

report_path = generate_firmwide_ec_report(
    aggregated_results=results, output_dir="econ_capital/reports"
)

time.sleep(1)  # Give filesystem time to update

report_pattern = str(
    root / "econ_capital" / "reports" / "firmwide" / "FirmWide_EC_Report_*.xlsx"
)
reports = glob.glob(report_pattern)

if reports:
    latest_report = max(reports, key=os.path.getctime)
    print(f"Opening latest report: {os.path.basename(latest_report)}")
    os.startfile(latest_report)  # Windows only
else:
    print("No report found in the reports directory.")

## Next Steps / Experiments

- Change `copula_df` in `generate_firmwide_ec_report()` (try 2.0 for fatter tails, 10+ for near-Gaussian)
- Modify correlations in `default.yaml` to observe diversification benefit changes
- Toggle WWR on/off in Credit Risk config and re-run
- Increase simulation paths to 1M+ for ultra-stable marginals
- Run sensitivity to confidence level (97.5% → 99.97%) by modifying `aggregate_economic_capital()` calls

See the other notebooks:
- `demo_credit.ipynb`
- `demo_market.ipynb`
- `demo_oprisk.ipynb`